In [5]:
from sqlalchemy import MetaData
from sqlalchemy import Table, Column, Integer, String, Float, text
from sqlalchemy import create_engine

import pandas as pd

from uuid import uuid4

from dotenv import load_dotenv

from sentence_transformers import CrossEncoder

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from qdrant_client import QdrantClient
from qdrant_client.http.models import VectorParams, Distance

from langchain_qdrant import QdrantVectorStore
from langchain_core.documents import Document
from langchain.tools import tool

import tqdm as notebook_tqdm

from langfuse import get_client
from langfuse.langchain import CallbackHandler
import os


load_dotenv()

True

In [6]:
langfuse = get_client()
langfuse_handler = CallbackHandler()

In [7]:
data_path = "/home/hasyim/Bootcamp_AI/capston/Capston3/chatbot/data/raw/imdb_top_1000.csv"
df = pd.read_csv(data_path)

In [8]:
df=df.replace({'Released_Year': 'PG'}, None)

In [9]:
df['Gross'] = df['Gross'].str.replace(',', '', regex=True)

In [10]:
df[['Released_Year','Gross']] = df[['Released_Year','Gross']].apply(pd.to_numeric)

In [11]:
df['film_id'] = [str(uuid4()) for _ in range(len(df['Series_Title']))]

In [12]:
df_clean = df.drop(columns="Overview")

In [42]:
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS FILM_TABEL"))

In [43]:
metadata_obj = MetaData()

In [41]:
engine = create_engine('sqlite:////home/hasyim/Bootcamp_AI/capston/Capston3/chatbot/data/process/capston3.db')

In [44]:
FILM = Table(
    "FILM_TABEL",
    metadata_obj,
    Column("film_id", String, primary_key=True, default=lambda: str(uuid4())),
    Column("Series_Title", String, nullable=False),
    Column("Released_Year", Float),
    Column("Certificate", String),
    Column("Runtime", String),
    Column("Genre", String),
    Column("IMDB_Rating", Float),
    Column("Meta_score", Float),
    Column("Director", String),
    Column("Star1", String),
    Column("Star2", String),
    Column("Star3", String),
    Column("Star4", String),
    Column("No_of_Votes", Integer),
    Column("Gross", Float),
    Column("Poster_Link", String),
    extend_existing=True
)

In [13]:
df_clean.to_sql(
    name='FILM_TABEL',
    con=engine,
    if_exists='append',
    index=False
)

NameError: name 'engine' is not defined

In [46]:
df_new = pd.read_sql('SELECT * FROM FILM_TABEL', con=engine)

In [9]:
df.head(1)

,Poster_Link,Series_Title,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Overview,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross,film_id
0,https://m.media-amazon.com/images/M/MV5BMDFkYT...,The Shawshank Redemption,1994.0,A,142 min,Drama,9.3,Two imprisoned men bond over a number of years...,80.0,Frank Darabont,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler,2343110,28341469.0,ce74ed01-ea88-4179-a0d1-0fc97c17556a


In [5]:
import subprocess

def check_gpu():
    try:
        # Menjalankan perintah nvidia-smi untuk cek koneksi GPU
        subprocess.check_output('nvidia-smi')
        return "cuda"
    except (subprocess.CalledProcessError, FileNotFoundError):
        return "cpu"

device = check_gpu()
print(f"Menggunakan: {device}")

Menggunakan: cuda


In [16]:
embedding = OpenAIEmbeddings(
    model='text-embedding-3-small',
)
# from langchain_huggingface import HuggingFaceEmbeddings
# import subprocess

# mask_trans = {
#     "device": check_gpu(),
#     "local_files_only": True
#               }
# embedding = HuggingFaceEmbeddings(
#     model="Qwen/Qwen3-Embedding-0.6B",
#     model_kwargs=mask_trans, 
#     cache_folder="chatbot/model_coba",
#     show_progress=True
#     )

rerank = CrossEncoder("Qwen/Qwen3-Reranker-0.6B", device='cuda', cache_folder="chatbot/model", local_files_only=True)

url = os.getenv("QDRANT_URL")

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 360.79it/s]
Default prompt name is set to 'query'. This prompt will be applied to all inference calls, except if a `prompt` or `prompt_name` parameter is provided.


In [31]:
documents = []

for i in range(len(df)):
    judul_film = df['Series_Title'][i]
    overview_film = df['Overview'][i]
    id_film = df['film_id'][i]
    input_rag = f"Series_Title: {judul_film}, Overview: {overview_film}"
    doc = Document(
        page_content=input_rag,
        metadata={
            "film_id": id_film,
            "Series_Title": judul_film
        },
    )
    documents.append(doc)

In [32]:
uuids = [str(uuid4()) for _ in range(len(documents))]

In [17]:
embedding

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x764dd817a3c0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x764dd817ad80>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [18]:
url = os.getenv("QDRANT_URL")

In [19]:
client = QdrantClient(
    url=url,
    api_key=os.getenv("QDRANT_API")
)


In [40]:
qdrant = QdrantVectorStore.from_documents(
    documents,
    embedding=embedding,
    url=url,
    api_key=os.getenv("QDRANT_API"),
    collection_name="coba_lokal_model_embedding",
)

In [20]:
client = QdrantClient(
    url=url,
    api_key=os.getenv("QDRANT_API")
)

response = client.get_collections()
print(response)

collections=[CollectionDescription(name='amazon_product'), CollectionDescription(name='coba_lokal_model_embedding'), CollectionDescription(name='flipkart_products'), CollectionDescription(name='data_capston3'), CollectionDescription(name='Data_IMDB')]


In [22]:
retrive = QdrantVectorStore.from_existing_collection(
    embedding=embedding,
    collection_name="Data_IMDB",
    url=url,
    api_key=os.getenv("QDRANT_API"),
)

In [23]:
query = "bisa rekomendasikan saya film tentang mafia?"
results = retrive.similarity_search(
    query=query,
    k=5,
)
for result in results:
    print(result)
    
hasil_rag = [result.page_content for result in results]

page_content='Series_Title: Donnie Brasco, Overview: An FBI undercover agent infiltrates the mob and finds himself identifying more with the mafia life, at the expense of his regular one.' metadata={'film_id': 'f1a910b7-3cc7-4b88-8266-19e1993bb654', 'Series_Title': 'Donnie Brasco', '_id': 'd37c8fce-5362-489d-b28a-721a6ed8a3cd', '_collection_name': 'Data_IMDB'}
page_content='Series_Title: Serbuan maut, Overview: A S.W.A.T. team becomes trapped in a tenement run by a ruthless mobster and his army of killers and thugs.' metadata={'film_id': '084ff6c8-aaf5-445a-af0c-25f4faaeec95', 'Series_Title': 'Serbuan maut', '_id': 'a9b0fae1-69d4-4dc7-8e9a-6cd3f2d09a1b', '_collection_name': 'Data_IMDB'}
page_content='Series_Title: American Gangster, Overview: An outcast New York City cop is charged with bringing down Harlem drug lord Frank Lucas, whose real life inspired this partly biographical film.' metadata={'film_id': 'a77acdf3-96c3-409a-a407-c5a0808eb1a4', 'Series_Title': 'American Gangster', '_i

In [51]:
reranking = rerank.rank(
    query, 
    hasil_rag,
    return_documents=True, 
    top_k=3
)

In [52]:
reranking

[{'corpus_id': 2,
  'score': np.float32(2.4375),
  'text': 'Series_Title: Scarface: The Shame of the Nation, Overview: An ambitious and nearly insane violent gangster climbs the ladder of success in the mob, but his weaknesses prove to be his downfall.'},
 {'corpus_id': 3,
  'score': np.float32(2.3125),
  'text': 'Series_Title: Donnie Brasco, Overview: An FBI undercover agent infiltrates the mob and finds himself identifying more with the mafia life, at the expense of his regular one.'},
 {'corpus_id': 4,
  'score': np.float32(0.25),
  'text': 'Series_Title: The Godfather: Part III, Overview: Follows Michael Corleone, now in his 60s, as he seeks to free his family from crime and find a suitable successor to his empire.'}]

In [51]:
context_list = [item['text'] for item in reranking]
context_list

['Series_Title: Goodfellas, Overview: The story of Henry Hill and his life in the mob, covering his relationship with his wife Karen Hill and his mob partners Jimmy Conway and Tommy DeVito in the Italian-American crime syndicate.',
 "Series_Title: The Godfather, Overview: An organized crime dynasty's aging patriarch transfers control of his clandestine empire to his reluctant son.",
 'Series_Title: Donnie Brasco, Overview: An FBI undercover agent infiltrates the mob and finds himself identifying more with the mafia life, at the expense of his regular one.']

In [52]:
context_for_llm = "\n\n".join(context_list)
print(context_for_llm)

Series_Title: Goodfellas, Overview: The story of Henry Hill and his life in the mob, covering his relationship with his wife Karen Hill and his mob partners Jimmy Conway and Tommy DeVito in the Italian-American crime syndicate.

Series_Title: The Godfather, Overview: An organized crime dynasty's aging patriarch transfers control of his clandestine empire to his reluctant son.

Series_Title: Donnie Brasco, Overview: An FBI undercover agent infiltrates the mob and finds himself identifying more with the mafia life, at the expense of his regular one.


In [60]:
@tool
def RAG_tool(query: str) -> str:
    """This tools is used to call data from Qdrant based on user query"""
    results = retrive.similarity_search(query=query, k=5)
    hasil_rag = [result.page_content for result in results]
    reranking = rerank.rank(
    query, 
    hasil_rag,
    return_documents=True, 
    top_k=3
    )
    context_list = [item['text'] for item in reranking]
    return context_list
    
tools = [RAG_tool]

In [56]:
print("✅ Tools berhasil dibuat:")
for t in tools:
    print(f"   🔧 {t.name}: {t.description[:60]}...")

✅ Tools berhasil dibuat:
   🔧 RAG_tool: This tools is used to call data from Qdrant based on user qu...


In [61]:
RAG_tool.invoke(query)

['Series_Title: Goodfellas, Overview: The story of Henry Hill and his life in the mob, covering his relationship with his wife Karen Hill and his mob partners Jimmy Conway and Tommy DeVito in the Italian-American crime syndicate.',
 "Series_Title: The Godfather, Overview: An organized crime dynasty's aging patriarch transfers control of his clandestine empire to his reluctant son.",
 'Series_Title: Donnie Brasco, Overview: An FBI undercover agent infiltrates the mob and finds himself identifying more with the mafia life, at the expense of his regular one.']

In [10]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

In [14]:
from langchain.agents import create_agent
from chatbot.tools.tool import tools

SYSTEM_PROMPT = """You are an agent for retrive the document of RAG from user query, your job is to take user query and see what user intention and rewrite to be more efficeint query.
If user query is in langunage other tahn english tranlate it first adn rewrite it before call the tool for RAG
"""

agent_app = create_agent(
    llm,
    tools,
    system_prompt=SYSTEM_PROMPT
)

print("✅ Agent berhasil dibuat!")
print(f"   Model  : {llm.model_name}")
print(f"   Tools  : {[t.name for t in tools]}")

✅ Agent berhasil dibuat!
   Model  : gpt-4o-mini
   Tools  : ['RAG_tool']


In [15]:
from IPython.display import Markdown, display

# Test 1: Pertanyaan Produk
print("=" * 60)
print("TEST 1: Rekomendasi Produk")
print("=" * 60)

response = agent_app.invoke(
    {"messages": [{"role": "user", "content": "Saya penasaran saya kan belajar sejarah disekolah, nah apakah ada perang mengenai perang dunia ke 2?"}]}
)
answer = response["messages"][-1].content
display(Markdown(answer))

TEST 1: Rekomendasi Produk


UnexpectedResponse: Unexpected Response: 404 (Not Found)
Raw response content:
b'{"status":{"error":"Not found: Collection `percobaan_capston3` doesn\'t exist!"},"time":0.000014648}'

In [1]:
import langchain
from langchain_community.utilities.sql_database import SQLDatabase

import sqlite3

from sqlalchemy import create_engine
from sqlalchemy import StaticPool

import requests

In [2]:
data_base = create_engine("sqlite://///home/hasyim/projects-ai-engineer/Capston3/chatbot/data/process/IMDB_FILM_capston3.db")

In [3]:
db = SQLDatabase(data_base)

In [6]:
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit
from langchain_deepseek import ChatDeepSeek
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv

load_dotenv()

# llm = ChatDeepSeek(
#     model="deepseek-v4-pro"
# )

llm = ChatOpenAI(
    model="gpt-4o-mini"
)

toolkit = SQLDatabaseToolkit(db=db, llm=llm)

In [7]:
from langchain_classic import hub

prompt_template = hub.pull("langchain-ai/sql-agent-system-prompt")

In [8]:
from langchain.agents import create_agent

agent = create_agent(
    llm,
    toolkit.get_tools(),
    system_prompt=prompt_template.format(dialect=db.dialect, top_k=5)
)

In [9]:
def masuk_query(query):
  events = agent.stream(
      {"messages": [("user", query)]},
      stream_mode="values"
  )

  for event in events:
    event["messages"][-1].pretty_print()

In [10]:
query_user = "what movie that release between year 1999 to 2010 and have rating above 8?"
masuk_query(query_user)

================================ Human Message =================================

what movie that release between year 1999 to 2010 and have rating above 8?


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [16]:
from langchain_huggingface import HuggingFaceEmbeddings
import subprocess

In [17]:
def check_gpu():
    try:
        subprocess.check_output('nvidia-smi')
        return "cuda"
    except (subprocess.CalledProcessError, FileNotFoundError):
        return "cpu"

In [18]:
device_used=check_gpu()

In [21]:
device_used = {"device": check_gpu()}

In [22]:
device_used

{'device': 'cuda'}

In [ ]:
embedding = HuggingFaceEmbeddings(
    model="Qwen/Qwen3-Embedding-0.6B",
    model_kwargs=device_used, 
    cache_folder="chatbot/model_coba",
    show_progress=True
    )

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 1182.99it/s]


In [35]:
embedding

HuggingFaceEmbeddings(model_name='Qwen/Qwen3-Embedding-0.6B', cache_folder='chatbot/model_coba', model_kwargs={'device': 'cuda'}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [4]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

load_dotenv()

True

In [9]:
embedding = OpenAIEmbeddings(
    model='text-embedding-3-small',
)

retrive = QdrantVectorStore.from_existing_collection(
    embedding=embedding,
    path="/home/hasyim/movegent/chatbot/data/process/qdrant",
    collection_name="Data_IMDB"
)



RuntimeError: Storage folder /home/hasyim/movegent/chatbot/data/process/qdrant is already accessed by another instance of Qdrant client. If you require concurrent access, use Qdrant server instead.

In [6]:
retrive